# Lab 06 — Transações na prática (DuckDB)

**Onde roda:** 🟢 Browser (JupyterLite). Execute célula a célula.

Objetivo: ver `BEGIN`/`COMMIT`/`ROLLBACK` funcionando — o "tudo ou nada" da atomicidade.

In [ ]:
try:
    import duckdb
except ModuleNotFoundError:
    import piplite; await piplite.install('duckdb'); import duckdb

con = duckdb.connect()
con.execute('CREATE TABLE contas(id INTEGER, saldo DOUBLE)')
con.execute('INSERT INTO contas VALUES (1, 500), (2, 100)')
con.execute('SELECT * FROM contas ORDER BY id').fetchall()

## 1. ROLLBACK: desfazer uma transação

In [ ]:
con.execute('BEGIN')
con.execute('UPDATE contas SET saldo = saldo - 100 WHERE id = 1')
con.execute('UPDATE contas SET saldo = saldo + 100 WHERE id = 2')
print('dentro da transação:', con.execute('SELECT * FROM contas ORDER BY id').fetchall())
con.execute('ROLLBACK')
print('após ROLLBACK:    ', con.execute('SELECT * FROM contas ORDER BY id').fetchall())

Repare: depois do `ROLLBACK`, os saldos **voltaram** ao original — como se nada tivesse acontecido.

## 2. COMMIT: confirmar de vez

In [ ]:
con.execute('BEGIN')
con.execute('UPDATE contas SET saldo = saldo - 100 WHERE id = 1')
con.execute('UPDATE contas SET saldo = saldo + 100 WHERE id = 2')
con.execute('COMMIT')
print('após COMMIT:      ', con.execute('SELECT * FROM contas ORDER BY id').fetchall())

## 3. Sua vez (mini-desafio)
A transferência acima (COMMIT) foi **atômica**: o total de dinheiro no sistema se conserva. Calcule o **saldo total** das contas e guarde em `total`. Verifique.

In [ ]:
total = con.execute('SELECT SUM(saldo) FROM contas').fetchone()[0]
total

In [ ]:
def verificar(x):
    try:
        assert abs(float(x) - 600.0) < 1e-6, 'O total deveria ser 600 (500+100) — o dinheiro se conserva.'
        print('\u2705 Correto! A atomicidade preservou o total: nada some, nada duplica.')
    except AssertionError as e:
        print('\u274c', e)

verificar(total)